# El Zen de Python, con ejemplos

**Unidad 1 · Semana 1 · Notebook 2 de 2**

Esta notebook se estudia **después** del curso acelerado. El Zen no es un
reglamento ni un examen de memoria: es una colección de criterios para conversar
sobre diseño, legibilidad y mantenimiento.

## Objetivos

- Interpretar varios aforismos del Zen en contexto.
- Comparar soluciones que funcionan, pero comunican intenciones diferentes.
- Practicar refactorizaciones pequeñas sin cambiar el comportamiento.

Tiempo sugerido: **25–35 minutos**.

## 1. Leer el Zen

Python incluye el texto como un pequeño *easter egg*. Al importar `this`, el Zen
se imprime en inglés. Quédate con las ideas, no con una traducción literal.

In [ ]:
import this

## 2. Explícito es mejor que implícito

Los valores misteriosos obligan a reconstruir el contexto. Nombrar una regla
hace visible la intención y facilita modificarla.

In [ ]:
scores = [0.91, 0.74, 0.88, 0.62]

# Difícil de interpretar: ¿qué significa 0.8?
seleccion_implicit = [score for score in scores if score >= 0.8]

# La regla ahora tiene nombre.
UMBRAL_CONFIANZA = 0.80
seleccion_explicit = [score for score in scores if score >= UMBRAL_CONFIANZA]

assert seleccion_implicit == seleccion_explicit
seleccion_explicit

## 3. Simple es mejor que complejo

La expresividad de Python puede producir código muy compacto. Compacto no
significa necesariamente claro. Divide una transformación cuando cada paso tiene
un significado útil.

In [ ]:
textos = ["  Hola mundo  ", "IA clara", "  datos  "]

# Demasiadas decisiones comprimidas en una expresión.
resultado_compacto = sorted({palabra.lower() for texto in textos for palabra in texto.strip().split() if len(palabra) > 2})

# Mismos resultados, pasos que se pueden inspeccionar.
palabras = []
for texto in textos:
    palabras.extend(texto.strip().lower().split())

palabras_relevantes = {palabra for palabra in palabras if len(palabra) > 2}
resultado_simple = sorted(palabras_relevantes)

assert resultado_compacto == resultado_simple
resultado_simple

## 4. Plano es mejor que anidado

Las validaciones tempranas reducen niveles de indentación. Este patrón se conoce
como *guard clause*: descartamos pronto los casos que no pueden continuar.

In [ ]:
def etiqueta_anidada(registro):
    if registro is not None:
        if "score" in registro:
            if registro["score"] >= 0.80:
                return registro.get("etiqueta")
    return None


def etiqueta_plana(registro):
    if not registro:
        return None
    if registro.get("score", 0) < 0.80:
        return None
    return registro.get("etiqueta")


ejemplo = {"etiqueta": "positivo", "score": 0.91}
assert etiqueta_anidada(ejemplo) == etiqueta_plana(ejemplo)
etiqueta_plana(ejemplo)

## 5. La legibilidad cuenta

Los nombres describen el dominio; los comentarios deberían explicar decisiones,
no traducir línea por línea lo que el código ya dice.

In [ ]:
# Funciona, pero exige descifrar nombres y posiciones.
d = [("m1", 0.7), ("m2", 0.9), ("m3", 0.82)]
r = [x[0] for x in d if x[1] >= 0.8]

# La estructura y los nombres comunican la intención.
evaluaciones = [
    {"modelo": "m1", "score": 0.70},
    {"modelo": "m2", "score": 0.90},
    {"modelo": "m3", "score": 0.82},
]
modelos_confiables = [
    evaluacion["modelo"]
    for evaluacion in evaluaciones
    if evaluacion["score"] >= UMBRAL_CONFIANZA
]

assert r == modelos_confiables
modelos_confiables

## 6. Los errores no deberían pasar silenciosamente

Ignorar cualquier excepción puede convertir un defecto visible en datos
incorrectos. Captura la excepción concreta y decide qué información necesita la
persona que operará el programa.

In [ ]:
def leer_entero_silencioso(valor):
    try:
        return int(valor)
    except Exception:
        return None


def leer_entero(valor):
    try:
        return int(valor)
    except (TypeError, ValueError) as error:
        print(f"Entrada inválida {valor!r}: {error}")
        return None


print(leer_entero_silencioso("tres"))
print(leer_entero("tres"))

## 7. Lo práctico vence a lo puro

No toda duplicación exige una abstracción y no toda tarea necesita una clase. La
solución apropiada depende del tamaño, la frecuencia de cambio y las personas
que mantendrán el código. El Zen ayuda a formular preguntas; no reemplaza el
juicio profesional.

Preguntas útiles:

- ¿La versión “elegante” se entiende con mayor rapidez?
- ¿Puedo probar cada parte importante de manera aislada?
- ¿La abstracción representa un concepto real o sólo reduce líneas?
- ¿Los errores conservan suficiente contexto para diagnosticarlos?

## 8. Reto de refactorización

La siguiente función mezcla nombres crípticos, anidación y un `except` demasiado
amplio. Reescríbela sin cambiar su contrato: debe devolver el promedio de scores
válidos y `None` cuando no exista ninguno.

In [ ]:
def f(xs):
    r = []
    for x in xs:
        try:
            if x:
                if "score" in x:
                    if float(x["score"]) >= 0:
                        r.append(float(x["score"]))
        except Exception:
            pass
    return sum(r) / len(r) if r else None


datos = [{"score": "0.9"}, None, {}, {"score": "mal"}, {"score": 0.7}]
print(f(datos))

In [ ]:
# Escribe aquí tu versión.
def promedio_scores_validos(registros):
    scores = []
    for registro in registros:
        if not registro or "score" not in registro:
            continue
        try:
            score = float(registro["score"])
        except (TypeError, ValueError):
            continue
        if score >= 0:
            scores.append(score)

    return sum(scores) / len(scores) if scores else None


assert promedio_scores_validos(datos) == f(datos)
promedio_scores_validos(datos)

## Cierre

El código pitónico no es el que utiliza más trucos del lenguaje; es el que hace
evidente su intención, conserva los errores importantes y resulta razonable de
mantener. A lo largo del curso volveremos a estos criterios al diseñar modelos de
datos, pipelines y proyectos reproducibles.